In [ ]:
# DASHBOARD NOTEBOOK
# Libraries
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json
import importlib
import runpy
import re
import gc

# GENERAL PARAMETERS
RAW_ROOT =  Path("data/raw_signal")
RAW_EMG_DIR   = RAW_ROOT / "emg"
RAW_BIA_DIR   = RAW_ROOT / "bia"
RAW_NIRS_DIR  = RAW_ROOT / "nirs"
RAW_MYOTON_DIR = RAW_ROOT / "myoton"

## LABELING PARAMETERS
# PROTOCOLE SEQUENCE ORDER 
SEQ_ORDER = "INIT WU REST0 MVC_REF REST1 SVC_REF REST2 EX_DYN REST3 MVC_RECOV_DYN REST4 EX_STA REST5 MVC_RECOV_STA END".split()


## 0. SUBJECT SELECTION

In [ ]:

# Set RUN_ID to the participan folder name (format: XXXNnPp_YYYYMMDD)
RUN_ID = ""


# 
# 
# 
# 
# 
# 
# 
# 


########

#PATH
PARTICIPANTS_DIR = Path("data/participants")  

# SET UP CACHE PATHS PARAMETERS
BASE_DERIVED = Path("data/derived") / RUN_ID
CACHE_DIR = BASE_DERIVED / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True) #create cache directory if it doesn't exist

# SET UP INITIAL CONTEXT CELL (CTX)
# CONTEXT  CELL

CTX = {
    # identity / paths
    "RUN_ID": RUN_ID,
    "CACHE_DIR": CACHE_DIR,
    "TORQUE_COL": "torque_raw",
}

## 1. PARTICIPANT INFO LOADER

In [ ]:
# 01 - PARTICIPANTS
import importlib
from sources import _01_participant
importlib.reload(_01_participant)

participants_df = _01_participant.run_participant(
    ctx=CTX,
    participants_dir=PARTICIPANTS_DIR,
    CACHE_DIR=CACHE_DIR,
    force_recompute=True,
)

CTX["participants_df"] = participants_df
participants_df

## 2. EMG LOADER

In [ ]:
# 02 - EMG IMPORT (function-based)

import importlib
from sources import _02_emg_import
importlib.reload(_02_emg_import)

master_index_grid, emg_compact_df, torque_compact_df, ts_ref = _02_emg_import.run_emg_import(
    ctx=CTX,
    raw_emg_dir=RAW_EMG_DIR,
    run_id=RUN_ID,
    participants_df=participants_df,
    force_recompute=False,
)

# fill ctx for downstream
CTX["master_index_grid"] = master_index_grid
CTX["emg_compact_df"] = emg_compact_df
CTX["torque_compact_df"] = torque_compact_df
CTX["ts_ref"] = ts_ref

## 03. LABELLING SEQUENCES

In [ ]:
# 03a - MANUAL SEQ PICKER (torque-based)
%matplotlib widget

from sources import _03a_sequence_picker
importlib.reload(_03a_sequence_picker)

output_json_path = _03a_sequence_picker.run_sequence_manual_selection(
    ctx=CTX,
    seq_order=SEQ_ORDER,
    force_recompute=True,
)


In [ ]:
# 03b - Apply SEQ labeling
%matplotlib inline
from sources import _03b_sequence_labeling
import importlib
importlib.reload(_03b_sequence_labeling)



master_index_grid, emg_compact_df, torque_compact_df = (
    _03b_sequence_labeling.run_apply_label_to_grid(
        ctx=CTX,
        seq_order=SEQ_ORDER,
        force_recompute=False,
    )
)

# update CTX with labeled versions
CTX["master_index_grid"] = master_index_grid
CTX["emg_compact_df"] = emg_compact_df
CTX["torque_compact_df"] = torque_compact_df

## 04. VOLUNTARY CONTRACTION DETECTION (VC)

In [ ]:
# 04 - VC DETECTION (TUNE)

from sources import _04_VC_detection
import importlib
importlib.reload(_04_VC_detection)

# Knobs by sequence for VC detection. Use these to optimize detection per sequence type, or leave uniform across sequences.
VC_KNOBS_BY_SEQ = {
    "WU":            {"thr_frac": 0.23, "merge_gap_s": 2, "min_dur_s": 2.00},
    "MVC_REF":       {"thr_frac": 0.20, "merge_gap_s": 0.10, "min_dur_s": 3.00},
    "SVC_REF":       {"thr_frac": 0.20, "merge_gap_s": 0.90, "min_dur_s": 3.00},
    "EX_DYN":        {"thr_frac": 0.20, "merge_gap_s": 0.25, "min_dur_s": 1.50},
    "MVC_RECOV_DYN": {"thr_frac": 0.20, "merge_gap_s": 0.10, "min_dur_s": 3.00},
    "EX_STA":        {"thr_frac": 0.20, "merge_gap_s": 0.10, "min_dur_s": 40.00},
    "MVC_RECOV_STA": {"thr_frac": 0.20, "merge_gap_s": 0.10, "min_dur_s": 3.00},
}


master_index_grid, torque_compact_df, emg_compact_df, vc_events_df = (
    _04_VC_detection.run_voluntary_contraction_detection(
        ctx=CTX,
        vc_knobs_by_seq=VC_KNOBS_BY_SEQ,
        force_recompute=True,
    )
)

CTX["master_index_grid"] = master_index_grid
CTX["torque_compact_df"] = torque_compact_df
CTX["emg_compact_df"] = emg_compact_df

In [ ]:
# 04b - VC COMMIT (upgrade canonical 02 files)

paths = CTX["parquet_path"]  # get paths from CTX (set in 02_emg_import)

for key in ["CACHE_MASTER", "CACHE_EMG", "CACHE_TORQUE"]:
    assert key in paths, f"Missing parquet path in CTX: {key}"

for col in ["VC", "VC_count"]:
    assert col in master_index_grid.columns, f"master_index_grid missing '{col}'"

master_index_grid.to_parquet(paths["CACHE_MASTER"], index=False)
emg_compact_df.to_parquet(paths["CACHE_EMG"], index=False)
torque_compact_df.to_parquet(paths["CACHE_TORQUE"], index=False)

print("[04_VC_detection] VC committed: canonical 02 cache files upgraded.")

## 5. BIA LOADER & SYNC

In [ ]:
# 05a - BIA IMPORT

from sources import _05a_bia_import
import importlib
importlib.reload(_05a_bia_import)


bia2_compact_df, bia4_compact_df, bia2_freqs_hz, bia4_freqs_hz = (
    _05a_bia_import.run_bia_import(
        ctx=CTX,
        raw_bia_dir=RAW_BIA_DIR,
        ts_ref=ts_ref,
        force_recompute=False,
    )
)

CTX["bia2_compact_df"] = bia2_compact_df
CTX["bia4_compact_df"] = bia4_compact_df
CTX["bia2_freqs_hz"] = bia2_freqs_hz
CTX["bia4_freqs_hz"] = bia4_freqs_hz

In [ ]:
# 05b. BIA SYNC - TUNE (function-based)

# knobs - EDIT THOSE AND RERUN CELL UNTIL SYNC LOOKS GOOD

BIA_SYNC_LAG_MIN_S  = -5
BIA_SYNC_LAG_MAX_S  = 5
BIA_SYNC_MANUAL_NUDGE_S = -0

# reload 
import importlib
import sources._05b_bia_sync as bia_sync_mod
importlib.reload(bia_sync_mod)

bia2_compact_aligned_df, bia4_compact_aligned_df, bia_sync_qc_fig_out = bia_sync_mod.run_bia_sync(
    ctx=CTX,
    lag_min_s=BIA_SYNC_LAG_MIN_S,
    lag_max_s=BIA_SYNC_LAG_MAX_S,
    manual_nudge_s=BIA_SYNC_MANUAL_NUDGE_S,
    torque_col=CTX.get("TORQUE_COL", "torque_raw"),
    force_recompute=True,
)

# ctx for downstream
CTX["bia2_compact_aligned_df"] = bia2_compact_aligned_df
CTX["bia4_compact_aligned_df"] = bia4_compact_aligned_df


In [ ]:
# 05b - BIA SYNC COMMIT (freeze current aligned data)


paths = CTX["parquet_path"]  # get paths from CTX 

for key in ["CACHE_BIA_2_ALIGNED", "CACHE_BIA_4_ALIGNED"]:
    assert key in paths, f"Missing parquet path in CTX: {key}"


bia2_compact_aligned_df.to_parquet(paths["CACHE_BIA_2_ALIGNED"], index=False)
bia4_compact_aligned_df.to_parquet(paths["CACHE_BIA_4_ALIGNED"], index=False)
pd.DataFrame({"freq_hz": bia2_freqs_hz}).to_parquet(paths["CACHE_BIA_2_FREQS"], index=False)
pd.DataFrame({"freq_hz": bia4_freqs_hz}).to_parquet(paths["CACHE_BIA_4_FREQS"], index=False)

qc_export_root = Path("results") / "QC_EXPORT"
qc_export_root.mkdir(parents=True, exist_ok=True)
bia_sync_qc_fig_path = qc_export_root / f"05b_bia_sync_{CTX['RUN_ID']}.png"
if bia_sync_qc_fig_out is not None:
    bia_sync_qc_fig_out.savefig(bia_sync_qc_fig_path, dpi=200, bbox_inches="tight")
    print(f"[05b_bia_sync] QC plot exported → {bia_sync_qc_fig_path}")
else:
    print("[05b_bia_sync] No QC figure in memory to export (cache hit in tune cell).")

print("[05b_bia_sync] BIA committed")




## 6. NIRS LOADER & SYNC

In [ ]:
# 06a - NIRS IMPORT

from sources import _06a_nirs_import
import importlib
importlib.reload(_06a_nirs_import)

nirs_compact_df = _06a_nirs_import.run_nirs_import(
    ctx=CTX,
    raw_nirs_dir=RAW_NIRS_DIR,
    ts_ref=ts_ref,
    force_recompute=False,
)

CTX["nirs_compact_df"] = nirs_compact_df

In [ ]:
# 6b. NIRS SYNC - TUNE (function-based)

from sources import _06b_nirs_sync
import importlib
importlib.reload(_06b_nirs_sync)

# knobs - EDIT THOSE AND RERUN CELL UNTIL SYNC LOOKS GOOD
NIRS_SYNC_TARGET_SEQ = "MVC_REF"
NIRS_SYNC_SIGNAL_COL = "nirs_hbdiff_tx1" #current scoring method works with nirs_hhb_tx1 and with nirs_hbdiff_tx1. Avoid o2 & thb with current method or edit in _06b_nirs_sync.py.
NIRS_SYNC_LAG_MIN_S = -5
NIRS_SYNC_LAG_MAX_S = 5
NIRS_MANUAL_NUDGE_S = 0
NIRS_SYNC_FORCE_RECOMPUTE = False  # new: ignore cache without deleting parquet

nirs_compact_aligned_df, nirs_sync_qc_fig_out = _06b_nirs_sync.run_nirs_sync(
    ctx=CTX,
    nirs_signal_col=NIRS_SYNC_SIGNAL_COL,
    lag_min_s=NIRS_SYNC_LAG_MIN_S,
    lag_max_s=NIRS_SYNC_LAG_MAX_S,
    manual_nudge_s=NIRS_MANUAL_NUDGE_S,
    torque_col=CTX.get("TORQUE_COL", "torque_raw"),
    force_recompute=True,
)

# keep in ctx for downstream steps too
CTX["nirs_compact_aligned_df"] = nirs_compact_aligned_df

# optional: naming 
nirs_sync_qc_fig_name = f"06b_nirs_sync_tune_{CTX['RUN_ID']}.png"



In [ ]:
# 06b - NIRS SYNC COMMIT (freeze current aligned data)

paths = CTX["parquet_path"]  # get paths from CTX 

for key in ["CACHE_NIRS_ALIGNED"]:
    assert key in paths, f"Missing parquet path in CTX: {key}"

nirs_compact_aligned_df.to_parquet(paths["CACHE_NIRS_ALIGNED"], index=False)

qc_export_root = Path("results") / "QC_EXPORT"
qc_export_root.mkdir(parents=True, exist_ok=True)
nirs_sync_qc_fig_path = qc_export_root / f"06b_nirs_sync_tune_{CTX['RUN_ID']}.png"
if nirs_sync_qc_fig_out is not None:
    nirs_sync_qc_fig_out.savefig(nirs_sync_qc_fig_path, dpi=200, bbox_inches="tight")
    print(f"[06b_nirs_sync] QC plot exported → {nirs_sync_qc_fig_path}")
else:
    print("[06b_nirs_sync] No QC figure in memory to export (cache hit in tune cell).")

print(f"[06b_nirs_sync] data committed → {paths['CACHE_NIRS_ALIGNED'].name}")



## 7. MYOTON LOADER & SYNC

In [ ]:
# 07 — MYOTON IMPORT (+ optional QC plot)

from sources import _07_myoton_import_sync
import importlib
importlib.reload(_07_myoton_import_sync)

myoton_compact_df, fig_myoton_qc = _07_myoton_import_sync.run_myoton_load_sync(
    ctx=CTX,
    raw_myoton_dir=RAW_MYOTON_DIR,
    ts_ref=ts_ref,
    force_recompute=False,
)

CTX["myoton_compact_df"] = myoton_compact_df

## 8. FINAL FULL QC PLOTS

In [ ]:
# 08 - FINAL QC CHECK (PLOTS ONLY)
%matplotlib inline

from sources import _08_final_quality_check
import importlib
importlib.reload(_08_final_quality_check)

#pick the plots you want to see in the final QC step. Available plots are: "overview_emg_torque", "bia_time", "bia_nyquist", "nirs_tx", "myoton". Note that some plots may not be available for all runs, depending on data quality and processing outcomes. If a selected plot is not available, it will be skipped with a warning message.
#They do cache from actual cached filed up to this point rather than global variables so it's probably a smart idea to plot them, just to check. 
PLOT_SELECTION = ["overview_emg_torque", 
                  "bia_time", 
                  "bia_nyquist", 
                  "nirs_tx", 
                  "myoton"
                  ]  

qc_figs = _08_final_quality_check.run_final_qc(
    ctx=CTX,
    plot_selection=PLOT_SELECTION,
)

for fig in qc_figs:
    display(fig)


## 9. DATA EXPORT

In [ ]:
# 9a. QC PDF EXPORT

from sources import _09_export
import importlib
importlib.reload(_09_export)

qc_pdf_path = _09_export.export_final_qc_pdf(
    qc_figs=qc_figs,
    run_id=RUN_ID,
)



In [ ]:
# 9b. MANUAL SEQUENCE VALIDITY REVIEW

from sources import _09_export
import importlib
importlib.reload(_09_export)

# default = keep all sequences unless manually overridden after QC review. 0 = bad, 1 = all good.
# EMG stays in one wide parquet, so its validity flags are split by sensor.
SEQUENCE_VALID_OVERRIDES = {
    "emg6": {}, # syntax : {sequence_name: validity_flag}. 
    "emg8": {},
    "emg10": {}, # example : mark all sequences as bad for emg10.
    "torque": {},
    "bia": {"INIT": 0}, # syntax : {sequence_name: validity_flag}
    "nirs": {"SVC_REF": 0}, # mark EVERYTHING in 0, it's a very corrupted file : manual review of what can be salvaged.
    "myoton": {},
}

sequence_valid_by_target = _09_export.build_sequence_valid_maps(
    ctx=CTX,
    sequence_valid_overrides=SEQUENCE_VALID_OVERRIDES,
)

print("sequence_valid mappings:")
for target_name, sequence_valid_by_seq in sequence_valid_by_target.items():
    print(f"\n[{target_name}]")
    for seq_name, keep_flag in sequence_valid_by_seq.items():
        print(f"- {seq_name}: {keep_flag}")



In [ ]:
# 9c. FINAL PARQUET EXPORT

from sources import _09_export
import importlib
importlib.reload(_09_export)

# pick the parquet files you want to export from the cache to results/DATA_EXPORT/<RUN_ID>
# use "all" to export the standard final set, or provide a list of file names from _09_export.PARQUET_EXPORT_LIST
DATA_EXPORT_SELECTION = "all"

data_export_root = _09_export.export_final_parquet(
    ctx=CTX,
    export_selection=DATA_EXPORT_SELECTION,
    sequence_valid_by_target=sequence_valid_by_target,
)



## 9d. EXPORTED SEQUENCE VALIDITY CHECK (TEMP)


In [ ]:
# Read every exported run folder and print only the sequences flagged as invalid.
# This is a read-only QC check on the saved exports, not on in-memory objects.

data_export_root = Path("results") / "DATA_EXPORT"
assert data_export_root.exists(), f"Missing export root: {data_export_root}"

run_export_dirs = sorted([p for p in data_export_root.iterdir() if p.is_dir()])
assert run_export_dirs, f"No exported run folders found in: {data_export_root}"

for run_export_dir in run_export_dirs:
    validity_files = {
        "emg": run_export_dir / "preprocessed_emg.parquet",
        "torque": run_export_dir / "preprocessed_torque.parquet",
        "bia2": run_export_dir / "preprocessed_bia2.parquet",
        "bia4": run_export_dir / "preprocessed_bia4.parquet",
        "nirs": run_export_dir / "preprocessed_nirs.parquet",
        "myoton": run_export_dir / "preprocessed_myoton.parquet",
    }

    missing_files = [file_key for file_key, file_path in validity_files.items() if not file_path.exists()]
    if missing_files:
        print(f"\n[{run_export_dir.name}] skipped - missing exported files: {missing_files}")
        continue

    print(f"\n[{run_export_dir.name}]")

    emg_export_df = pd.read_parquet(
        validity_files["emg"],
        columns=["SEQ", "sequence_valid_emg6", "sequence_valid_emg8", "sequence_valid_emg10"],
    )
    for validity_col in ["sequence_valid_emg6", "sequence_valid_emg8", "sequence_valid_emg10"]:
        invalid_seq_list = sorted(
            emg_export_df.loc[emg_export_df[validity_col] == 0, "SEQ"].dropna().astype(str).unique().tolist()
        )
        print(f"- {validity_col}: {invalid_seq_list if invalid_seq_list else 'none'}")

    for file_key in ["torque", "bia2", "bia4", "nirs", "myoton"]:
        export_df = pd.read_parquet(validity_files[file_key], columns=["SEQ", "sequence_valid"])
        invalid_seq_list = sorted(
            export_df.loc[export_df["sequence_valid"] == 0, "SEQ"].dropna().astype(str).unique().tolist()
        )
        print(f"- {file_key}: {invalid_seq_list if invalid_seq_list else 'none'}")


## 10. UTILS

One-off utilities parked outside the main pipeline flow. 
You probably don't need this ; some DB integration requiere a participant metadata files including every subject. Leaving it here, do as you wish. Uncomment to use.


In [ ]:
# 10a. EXPORT ALL PARTICIPANTS (ONE-OFF UTILITY)
# This does not change the main pipeline participant logic.

# from sources.utils.export_all_participants import export_all_participants

# participants_all_path = export_all_participants()
# participants_all_df = pd.read_parquet(participants_all_path)

# print(f"participants_all.parquet written to: {participants_all_path}")
# print(f"Rows: {len(participants_all_df)}")
# print(f"Columns: {list(participants_all_df.columns)}")
# participants_all_df.head()
